In [0]:
orders_bronze = spark.read.format("delta").load("/Volumes/workspace/default/raw_uploads/bronze/orders")
customers_df = spark.read.csv(
    "/Volumes/workspace/default/raw_uploads/olist_customers_dataset.csv",
    header=True,
    inferSchema=True,
)

from pyspark.sql import functions as F

#were starting with this because silver alwayds reads from vronze, so if there was an error in the silver, you can reprocess from bronze instead of from strach

In [0]:
#find which order_ids are duplicated
duplicate_ids = (
    orders_bronze.groupBy("order_id")
    .agg(F.count("*").alias("row_count"))
    .filter(F.col("row_count") > 1)
    .select("order_id")
)

#quarantine: the actual duplicate rows, kept for inspection
quarantined_duplicates = orders_bronze.join(duplicate_ids, on="order_id", how="inner")

#clean: keep only one row per order_id going forward
orders_deduped = orders_bronze.dropDuplicates(["order_id"])

print(f"Quarantined duplicate rows: {quarantined_duplicates.count()}")
print(f"Orders after dedupL {orders_deduped.count()}")

#.dropDuplicates(["order_id"]) - a new method that keeps only first row it encounters for each unique order_id, and throws away the rest.
#** note the list ["order_id"] is telling spark to consider row duplicates baside on this column specifically

#why quarantine instead of j dropping: we dont want bad records to j vanish w no trace, we want to keep them in a separate table so we can go looj at what we excluded and why


In [0]:
orphaned_orders = orders_deduped.join(customers_df, on="customer_id", how="left_anti")
orders_with_valid_customer = orders_deduped.join(customers_df, on="customer_id", how="inner")

print(f"Orphaned orders (quarantined): {orphaned_orders.count()}")
print(f"Orders with valid customer: {orders_with_valid_customer.count()}")

#we used inner join here instead of left join because we want to keep only those orders that have a valid customer, not all orders
#inner join will keep only those rows that have a match in both tables, left join will keep all rows from the left table and only those from the right table that have a match


In [0]:
null_status_orders = orders_with_valid_customer.filter(F.col("order_status").isNull())
orders_clean_status = orders_with_valid_customer.filter(F.col("order_status").isNotNull())

print(f"Null status orders(quarantined): {null_status_orders.count()}")
print(f"Orders with valid status: {orders_clean_status.count()}")

#isNotNull() - keeps only rows where the column does have a value

In [0]:
#standardized types and add a clean date column

orders_silver = orders_clean_status.withColumn(
    "order_date", F.to_date("order_purchase_timestamp")
).withColumn(
    "order_status", F.lower(F.trim(F.col("order_status")))
)

orders_silver.select("order_id", "order_status", "order_date").show(5)

#F.trim - removes anu accidental leading/trailing whitespace from a text value
#F.lower- converts all text into lowercase... this matters bc wo it groupBy("order_status") could accidentally treat those as three diff categories instead of one

#standardizing text values is important steps of silver layer work, makes it more optimized and easier for later steps

In [0]:
# The clean, validated table
orders_silver.write.format("delta").mode("overwrite").save(
    "/Volumes/workspace/default/raw_uploads/silver/orders"
)

# The quarantine tables - kept separately for auditing, not deleted
quarantined_duplicates.write.format("delta").mode("overwrite").save(
    "/Volumes/workspace/default/raw_uploads/silver_quarantine/duplicate_orders"
)
orphaned_orders.write.format("delta").mode("overwrite").save(
    "/Volumes/workspace/default/raw_uploads/silver_quarantine/orphaned_orders"
)
null_status_orders.write.format("delta").mode("overwrite").save(
    "/Volumes/workspace/default/raw_uploads/silver_quarantine/null_status_orders"
)

print("Silver orders table and quarantine tables written.")

#why we are writing quarantine tables as separate delta tables, not just printing counts: this is what makes the quarentine tables useful rather than j theorhetical, anyone would be able to query silver_quarantine/orphaned_orders and see which rows were excluded and why

In [0]:
print(f"Bronze orders total: {orders_bronze.count()}")
print(f"Duplicates removed: {quarantined_duplicates.count()}")
print(f"Orphaned removed: {orphaned_orders.count()}")
print(f"Null status removed: {null_status_orders.count()}")
print(f"Final Silver orders: {orders_silver.count()}")

In [0]:
#loading all tables needed for today
customers_df = spark.read.csv(
    "/Volumes/workspace/default/raw_uploads/olist_customers_dataset.csv",
    header=True,
    inferSchema=True,
)

order_items_df = spark.read.csv(
    "/Volumes/workspace/default/raw_uploads/olist_order_items_dataset.csv",
    header=True,
    inferSchema=True,
)

order_payments_df = spark.read.csv(
    "/Volumes/workspace/default/raw_uploads/olist_order_payments_dataset.csv",
    header=True,
    inferSchema=True,
)

customers_df.printSchema()
order_items_df.printSchema()
order_payments_df.printSchema()

#we need these tables bc our gold layer is going to answer questions about sales and revenue

In [0]:
#cleaning up the customers table

from pyspark.sql import functions as F

# check for duplicates first
duplicate_customers = (
    customers_df.groupBy("customer_id")
    .agg(F.count("*").alias("row_count"))
    .filter(F.col("row_count") > 1)
)
print(f"Duplicate customer_ids: {duplicate_customers.count()}")

# standardize text fields the same way we did for order_status
customers_silver = customers_df.withColumn(
    "customer_city", F.lower(F.trim(F.col("customer_city")))
).withColumn(
    "customer_state", F.upper(F.trim(F.col("customer_state")))
).dropDuplicates(["customer_id"])

customers_silver.select("customer_id", "customer_city", "customer_state").show(5)



In [0]:
# do all order_items reference a real order?
orphaned_items = order_items_df.join(orders_silver, on="order_id", how="left_anti")
print(f"Order items with no matching order: {orphaned_items.count()}")

#do all order_payments reference a real order?
orphaned_payments = order_payments_df.join(orders_silver, on="order_id", how="left_anti")
print(f"Payments with no matching order: {orphaned_payments.count()}")

In [0]:
#item prices that are zero or negative don't make business sense
bad_prices = order_items_df.filter(F.col("price") <= 0)
print(f"Order items with price <= 0: {bad_prices.count()}")

#payment values that are zero or negative
bad_payments = order_payments_df.filter(F.col("payment_value") <= 0)
print(f"Payments with value <= 0: {bad_payments.count()}")



In [0]:
items_silver = order_items_df.join(
    orders_silver.select("order_id"), on="order_id", how="inner"
).filter(F.col("price") > 0)

payments_silver = order_payments_df.join(
    orders_silver.select("order_id"), on="order_id", how="inner"
).filter(F.col("payment_value") > 0)

print(f"Order items - Silver: {items_silver.count()}")
print(f"Payments - Silver: {payments_silver.count()}")

#orders_silver.select("order_id") - were only selecting the order_id from orders_silver before joining, rather than the whole table. since we are only confirming "does this order_if exits" oulling every other column would create duplicate columns in the result for no reason

In [0]:
customers_silver.write.format("delta").mode("overwrite").save(
    "/Volumes/workspace/default/raw_uploads/silver/customers"
)
items_silver.write.format("delta").mode("overwrite").save(
    "/Volumes/workspace/default/raw_uploads/silver/order_items"
)
payments_silver.write.format("delta").mode("overwrite").save(
    "/Volumes/workspace/default/raw_uploads/silver/order_payments"
)

# Quarantine tables too, same pattern as yesterday
orphaned_items.write.format("delta").mode("overwrite").save(
    "/Volumes/workspace/default/raw_uploads/silver_quarantine/orphaned_order_items"
)
orphaned_payments.write.format("delta").mode("overwrite").save(
    "/Volumes/workspace/default/raw_uploads/silver_quarantine/orphaned_payments"
)

print("Silver customers, order_items, and order_payments tables written.")

In [0]:
print("=== Silver Layer Summary ===")
print(f"Orders: {orders_silver.count()}")
print(f"Customers: {customers_silver.count()}")
print(f"Order items: {items_silver.count()}")
print(f"Payments: {payments_silver.count()}")

In [0]:
# SILVER LAYER - CONSOLIDATED DATA QUALITY SUMMARY

print("=== SILVER TABLES (clean, validated) ===")
print(f"Orders:         {orders_silver.count()}")
print(f"Customers:      {customers_silver.count()}")
print(f"Order items:    {items_silver.count()}")
print(f"Order payments: {payments_silver.count()}")

print("\n=== QUARANTINED RECORDS (excluded, but preserved for audit) ===")
print(f"Duplicate orders:        {quarantined_duplicates.count()}")
print(f"Orphaned orders:         {orphaned_orders.count()}")
print(f"Null-status orders:      {null_status_orders.count()}")
print(f"Orphaned order items:    {orphaned_items.count()}")
print(f"Orphaned payments:       {orphaned_payments.count()}")

print("\n=== SOURCE COMPARISON (Bronze vs Silver) ===")
print(f"Bronze orders total:  {orders_bronze.count()}")
print(f"Silver orders total:  {orders_silver.count()}")
print(f"Rows excluded:        {orders_bronze.count() - orders_silver.count()}")

Here's the plain-English rundown of what you actually learned/did today (Day 4).

## What you built
- Cleaned up the **customers** table — removed duplicates, made city names lowercase and state codes uppercase so they're consistent.
- Loaded two brand new tables: **order_items** (what was in each order, including price) and **order_payments** (how much was paid, how).
- Checked both new tables for "orphan" records — rows pointing to an order that doesn't actually exist.
- Checked for prices/payments that were **zero or negative** — since that's basically never a real, valid value.
- Saved clean versions of all these tables, and kept the "bad" records in separate quarantine tables instead of just deleting them.

## Concepts to remember

- **Not all data quality checks are the same type.** So far you've done:
  - "Is something missing?" (null checks)
  - "Does this thing exist somewhere else it's supposed to?" (orphan/join checks)
  - **New today:** "Does this number actually make sense?" (a price of $0 or -$5 is nonsensical, even if it's technically "present")

- **Standardizing text isn't one-size-fits-all.** You made city names lowercase, but state codes uppercase — because that matches how each is actually written in the real world. The lesson: think about what the "correct" format for a field *should* look like, not just pick one rule and slap it on everything.

- **Quarantine, don't delete.** Every time you found bad data, you saved it somewhere else instead of just throwing it away — so if anyone ever asks "wait, why is this row missing," you can actually show them, instead of shrugging.

- **Build on top of already-cleaned data, not raw data.** When checking if order_items/payments had valid orders, you checked them against your *already-cleaned* orders table (Silver), not the original messy one (Bronze). Cleaning builds in layers — each one trusts the one before it.

- **Only pull in what you need before joining.** When joining just to check "does this ID exist," you only grabbed the ID column, not the entire other table — keeps things tidy and avoids clutter you don't need.

## Mistake/lesson to remember for next time
- **Variables don't carry over between different notebooks** — even in the same Databricks account, each notebook is its own separate "memory." This is why `orders_silver` kept throwing errors today — it existed in a different notebook, not this one. Going forward: keep Silver work in **one notebook**, not scattered across several.


=== SILVER TABLES (clean, validated) ===
Orders:         99441
Customers:      99441
Order items:    112650
Order payments: 103877

=== QUARANTINED RECORDS (excluded, but preserved for audit) ===
Duplicate orders:        198884
Orphaned orders:         0
Null-status orders:      0
Orphaned order items:    0
Orphaned payments:       0

=== SOURCE COMPARISON (Bronze vs Silver) ===
Bronze orders total:  198884
Silver orders total:  99441
Rows excluded:        99443